# ReelCraft — Free Google Colab GPU Runner

Run the full ReelCraft pipeline (Transcription, OpenCV YuNet Eye-Line Analysis, and HyperFrames Render) on a free Colab T4 GPU with zero local setup.

In [ ]:
# 1. Install system tools (Node 22, FFmpeg, OpenCV, yt-dlp)
!apt-get -qq update && apt-get -qq install -y ffmpeg curl
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get -qq install -y nodejs
!pip install -q opencv-python-headless numpy yt-dlp
!node -v && ffmpeg -version | head -n 1

In [ ]:
# 2. Clone ReelCraft
REPO_URL = "https://github.com/dvlprlat/ReelCraft.git"
!git clone {REPO_URL} /content/reelcraft
%cd /content/reelcraft
!npm install --silent
!npx hyperframes browser ensure

In [ ]:
# 3. Configure optional free Groq API key for 2-second Whisper transcription
import os
os.environ["GROQ_API_KEY"] = ""  # Paste free key from https://console.groq.com
!npm run doctor

In [ ]:
# 4. Upload your raw video or voiceover into input/demo/
from google.colab import files
import os, shutil
os.makedirs("input/demo", exist_ok=True)
uploaded = files.upload()
for name in uploaded.keys():
    shutil.move(name, f"input/demo/{name}")
    print("Saved to", f"input/demo/{name}")

In [ ]:
# 5. Transcribe & verify (replace raw.mp4 with your uploaded filename)
!node scripts/transcribe.mjs input/demo/raw.mp4 --out=work/demo/raw --provider=groq

In [ ]:
# 6. Render your HyperFrames project & download final MP4
!HYPERFRAMES_CHROME_SANDBOX=0 npx hyperframes render work/demo/hyperframes --quality looks --fps 30 -o out/demo.mp4
!node scripts/verify-render.mjs out/demo.mp4
from google.colab import files
files.download("out/demo.mp4")